In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
!pip install -q transformers datasets evaluate seqeval accelerate sentencepiece

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "tner/ontonotes5",
    revision="refs/convert/parquet"
)

print(dataset)

ontonotes5/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.68MB            

ontonotes5/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/515k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/517k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8262
    })
})


In [4]:
print(dataset)
print(dataset["train"].info)
print(dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8262
    })
})
DatasetInfo(features={'tokens': List(Value('string')), 'tags': List(Value('int32'))}, builder_name='parquet', dataset_name='ontonotes5', config_name='default', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=13964727, num_examples=59924, dataset_name='ontonotes5'), 'validation': SplitInfo(name='validation', num_bytes=1892580, num_examples=8528, dataset_name='ontonotes5'), 'test': SplitInfo(name='test', num_bytes=1953338, num_examples=8262, dataset_name='ontonotes5')}, download_size=4712334, dataset_size=17810645, size_in_bytes=22522979)
{'tokens': List(Value('string')), 'tags': List(Value('int32'))}


In [5]:
from datasets import get_dataset_config_names

print(dataset["train"].info.description)
print(dataset["train"].info.features)


{'tokens': List(Value('string')), 'tags': List(Value('int32'))}


In [6]:
id2label = {
    0: "O",
    1: "B-CARDINAL",
    2: "B-DATE",
    3: "I-DATE",
    4: "B-PERSON",
    5: "I-PERSON",
    6: "B-NORP",
    7: "B-GPE",
    8: "I-GPE",
    9: "B-LAW",
    10: "I-LAW",
    11: "B-ORG",
    12: "I-ORG",
    13: "B-PERCENT",
    14: "I-PERCENT",
    15: "B-ORDINAL",
    16: "B-MONEY",
    17: "I-MONEY",
    18: "B-WORK_OF_ART",
    19: "I-WORK_OF_ART",
    20: "B-FAC",
    21: "B-TIME",
    22: "I-CARDINAL",
    23: "B-LOC",
    24: "B-QUANTITY",
    25: "I-QUANTITY",
    26: "I-NORP",
    27: "I-LOC",
    28: "B-PRODUCT",
    29: "I-TIME",
    30: "B-EVENT",
    31: "I-EVENT",
    32: "I-FAC",
    33: "B-LANGUAGE",
    34: "I-PRODUCT",
    35: "I-ORDINAL",
    36: "I-LANGUAGE",
}

label2id = {label: idx for idx, label in id2label.items()}

print("Total labels:", len(id2label))
print("Labels:", list(id2label.values()))

Total labels: 37
Labels: ['O', 'B-CARDINAL', 'B-DATE', 'I-DATE', 'B-PERSON', 'I-PERSON', 'B-NORP', 'B-GPE', 'I-GPE', 'B-LAW', 'I-LAW', 'B-ORG', 'I-ORG', 'B-PERCENT', 'I-PERCENT', 'B-ORDINAL', 'B-MONEY', 'I-MONEY', 'B-WORK_OF_ART', 'I-WORK_OF_ART', 'B-FAC', 'B-TIME', 'I-CARDINAL', 'B-LOC', 'B-QUANTITY', 'I-QUANTITY', 'I-NORP', 'I-LOC', 'B-PRODUCT', 'I-TIME', 'B-EVENT', 'I-EVENT', 'I-FAC', 'B-LANGUAGE', 'I-PRODUCT', 'I-ORDINAL', 'I-LANGUAGE']


In [7]:
for example in dataset["train"]:
    if any(tag != 0 for tag in example["tags"]):
        for token, tag_id in zip(example["tokens"], example["tags"]):
            print(f"{token:20} -> {id2label[tag_id]}")
        break

But                  -> O
a                    -> O
chance               -> O
to                   -> O
fill                 -> O
out                  -> O
sales                -> O
-                    -> O
tax                  -> O
records              -> O
is                   -> O
rarely               -> O
one                  -> B-CARDINAL
of                   -> O
them                 -> O
.                    -> O


In [8]:
required_types = ["DATE", "PERSON", "PLACE", "ITEM", "EVENT"]

available_types = {
    label[2:]
    for label in id2label.values()
    if label.startswith(("B-", "I-"))
}

for entity_type in required_types:
    if entity_type == "PLACE":
        available = "GPE" in available_types or "LOC" in available_types
    elif entity_type == "ITEM":
        available = "PRODUCT" in available_types
    else:
        available = entity_type in available_types

    print(f"{entity_type}: {'Supported' if available else 'Needs custom handling'}")

DATE: Supported
PERSON: Supported
PLACE: Supported
ITEM: Supported
EVENT: Supported


In [9]:
count = 0

for example in dataset["train"]:
    for token, tag_id in zip(example["tokens"], example["tags"]):
        if id2label[tag_id] == "B-CARDINAL":
            print("Sentence:", " ".join(example["tokens"]))
            print("Entity:", token)
            print("-" * 60)

            count += 1
            if count == 5:
                break

    if count == 5:
        break

Sentence: But a chance to fill out sales - tax records is rarely one of them .
Entity: one
------------------------------------------------------------
Sentence: One senior executive at his firm spends nearly 20 % of his time on insurance , he says .
Entity: One
------------------------------------------------------------
Sentence: Eastern Reproduction Corp. , a Waltham , Mass. , maker of thin metal precision parts , must report to five federal and state agencies as well as to local fire , police , hospital and plumbing authorities , says Robert Maguire , president .
Entity: five
------------------------------------------------------------
Sentence: One state environmental regulator returned a report because `` it was n't heavy enough , it could n't have been correct , '' Mr. Maguire says .
Entity: One
------------------------------------------------------------
Sentence: Alice Fixx , who runs her own public - relations concern in New York , says she has had to overhaul her pension and

In [10]:
!pip install -q "huggingface_hub>=0.25.0,<2.0"

In [11]:
import datasets
import huggingface_hub

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)


datasets: 4.8.5
huggingface_hub: 1.33.0


In [12]:
from huggingface_hub import HfApi

model_id = "tner/deberta-v3-large-ontonotes5"
info = HfApi().model_info(model_id)

print("Model:", info.modelId)
print("Architecture:", info.config.get("architectures"))
print("Labels:", info.config.get("id2label"))

Model: tner/deberta-v3-large-ontonotes5
Architecture: ['DebertaV2ForTokenClassification']
Labels: None


In [13]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_id = "tner/deberta-v3-large-ontonotes5"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForTokenClassification.from_pretrained(model_id)

print("Tokenizer loaded:", tokenizer is not None)
print("Model loaded:", model is not None)
print("Number of labels:", model.config.num_labels)

config.json:   0%|          | 0.00/65.9k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.74GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Tokenizer loaded: True
Model loaded: True
Number of labels: 37


In [14]:
import torch
text = (
    "Last Sunday, Binu visited his daughter Anisha "
    "in Bhaktapur and celebrated her birthday."
)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = model(**inputs)

predictions = outputs.logits.argmax(dim=-1)[0]

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, pred_id in zip(tokens, predictions):
    label = model.config.id2label[pred_id.item()]
    print(f"{token:20} {label}")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.74GB            

model.safetensors: downloading bytes:           |  0.00B            

[CLS]                O
▁Last                B-DATE
▁Sunday              I-DATE
,                    O
▁Bin                 B-PERSON
u                    I-PERSON
▁visited             O
▁his                 O
▁daughter            O
▁An                  B-PERSON
isha                 I-PERSON
▁in                  O
▁Bha                 B-GPE
kta                  I-GPE
pur                  I-GPE
▁and                 O
▁celebrated          O
▁her                 O
▁birthday            O
.                    O
[SEP]                O


In [15]:

import torch

LABEL_MAP = {
    "PERSON": "PERSON",
    "DATE": "DATE",
    "GPE": "PLACE",
    "LOC": "PLACE",
    "FAC": "PLACE",
    "PRODUCT": "ITEM",
    "EVENT": "EVENT",
}

def extract_entities(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation="longest_first",
        return_offsets_mapping=True
    )

    # Offset mappings are metadata for aligning tokens to original text.
    offsets = inputs.pop("offset_mapping")[0].tolist()

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_ids = outputs.logits.argmax(dim=-1)[0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0].tolist()
    )

    entities = []
    current_start = None
    current_end = None
    current_type = None

    def save_entity():
        if current_start is not None and current_type is not None:
            entity_type = LABEL_MAP.get(current_type)

            if entity_type:
                entity_text = text[current_start:current_end].strip()

                if entity_text:
                    entities.append({
                        "type": entity_type,
                        "text": entity_text
                    })

    for token, pred_id, (start, end) in zip(
        tokens, predicted_ids, offsets
    ):
        if token in tokenizer.all_special_tokens or start == end:
            save_entity()
            current_start = None
            current_end = None
            current_type = None
            continue

        label = model.config.id2label[pred_id]

        if label == "O" or "-" not in label:
            save_entity()
            current_start = None
            current_end = None
            current_type = None
            continue

        prefix, entity_type = label.split("-", 1)

        if prefix == "B" or entity_type != current_type:
            save_entity()
            current_start = start
            current_end = end
            current_type = entity_type
        elif prefix == "I" and current_start is not None:
            current_end = end

    save_entity()
    return entities


In [16]:
text = (
    "Last Sunday, Binu visited his daughter Anisha "
    "in Bhaktapur and celebrated her birthday."
)

entities = extract_entities(text)

for entity in entities:
    print(entity)

{'type': 'DATE', 'text': 'Last Sunday'}
{'type': 'PERSON', 'text': 'Binu'}
{'type': 'PERSON', 'text': 'Anisha'}
{'type': 'PLACE', 'text': 'Bhaktapur'}


In [17]:

text = (
    "Last Sunday, Binu visited his daughter Anisha "
    "in Bhaktapur and celebrated her birthday."
)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation="longest_first"
)

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

predicted_ids = outputs.logits.argmax(dim=-1)[0].tolist()
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0].tolist()
)

for token, pred_id in zip(tokens, predicted_ids):
    label = model.config.id2label[pred_id]
    print(f"{token:20} {label}")


[CLS]                O
▁Last                B-DATE
▁Sunday              I-DATE
,                    O
▁Bin                 B-PERSON
u                    I-PERSON
▁visited             O
▁his                 O
▁daughter            O
▁An                  B-PERSON
isha                 I-PERSON
▁in                  O
▁Bha                 B-GPE
kta                  I-GPE
pur                  I-GPE
▁and                 O
▁celebrated          O
▁her                 O
▁birthday            O
.                    O
[SEP]                O


In [18]:

import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU — training will be slower.")

print("\nCurrent model:", model.config._name_or_path)
print("Current number of labels:", model.config.num_labels)
print("Current label mapping:")

for label_id, label_name in model.config.id2label.items():
    print(label_id, ":", label_name)


GPU available: True
GPU: Tesla T4

Current model: tner/deberta-v3-large-ontonotes5
Current number of labels: 37
Current label mapping:
0 : O
1 : B-CARDINAL
2 : B-DATE
3 : I-DATE
4 : B-PERSON
5 : I-PERSON
6 : B-NORP
7 : B-GPE
8 : I-GPE
9 : B-LAW
10 : I-LAW
11 : B-ORG
12 : I-ORG
13 : B-PERCENT
14 : I-PERCENT
15 : B-ORDINAL
16 : B-MONEY
17 : I-MONEY
18 : B-WORK_OF_ART
19 : I-WORK_OF_ART
20 : B-FAC
21 : B-TIME
22 : I-CARDINAL
23 : B-LOC
24 : B-QUANTITY
25 : I-QUANTITY
26 : I-NORP
27 : I-LOC
28 : B-PRODUCT
29 : I-TIME
30 : B-EVENT
31 : I-EVENT
32 : I-FAC
33 : B-LANGUAGE
34 : I-PRODUCT
35 : I-ORDINAL
36 : I-LANGUAGE


In [19]:
##actual finetuning after compeltion of data loading begin


In [20]:

from collections import Counter

tag_counts = Counter()

for example in dataset["train"]:
    for tag_id in example["tags"]:
        tag_counts[model.config.id2label[tag_id]] += 1

for label, count in tag_counts.most_common():
    print(f"{label:20} {count}")


O                    939111
I-ORG                18246
B-PERSON             15429
B-GPE                15405
I-DATE               13333
B-ORG                12820
I-PERSON             11147
B-DATE               10922
B-CARDINAL           7355
B-NORP               6870
I-MONEY              4912
I-GPE                3679
I-PERCENT            2498
B-MONEY              2411
I-WORK_OF_ART        2400
I-CARDINAL           2289
B-PERCENT            1763
B-ORDINAL            1640
I-EVENT              1605
B-LOC                1514
I-TIME               1507
I-FAC                1467
I-LOC                1395
I-QUANTITY           1235
B-TIME               1233
B-WORK_OF_ART        974
B-FAC                860
I-LAW                785
B-EVENT              748
B-QUANTITY           657
B-PRODUCT            606
I-PRODUCT            576
I-NORP               446
B-LANGUAGE           304
B-LAW                282
I-LANGUAGE           13
I-ORDINAL            5


In [21]:
label_mapping = {
    "PERSON": "PERSON",
    "GPE": "PLACE",
    "LOC": "PLACE",
    "FAC": "PLACE",
    "DATE": "DATE",
    "TIME": "TIME",
    "EVENT": "EVENT",
    "PRODUCT": "ITEM",
    "ORG": "ORGANIZATION",
}

entity_types = [
    "PERSON",
    "PLACE",
    "DATE",
    "TIME",
    "EVENT",
    "ITEM",
    "ORGANIZATION",
]

label_list = ["O"]

for entity in entity_types:
    label_list.extend([f"B-{entity}", f"I-{entity}"])

label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

print("Number of labels:", len(label_list))
print(label_list)

Number of labels: 15
['O', 'B-PERSON', 'I-PERSON', 'B-PLACE', 'I-PLACE', 'B-DATE', 'I-DATE', 'B-TIME', 'I-TIME', 'B-EVENT', 'I-EVENT', 'B-ITEM', 'I-ITEM', 'B-ORGANIZATION', 'I-ORGANIZATION']


In [22]:
print("Original label IDs:")
for label_id, label in sorted(model.config.id2label.items()):
    print(label_id, "->", label)

Original label IDs:
0 -> O
1 -> B-CARDINAL
2 -> B-DATE
3 -> I-DATE
4 -> B-PERSON
5 -> I-PERSON
6 -> B-NORP
7 -> B-GPE
8 -> I-GPE
9 -> B-LAW
10 -> I-LAW
11 -> B-ORG
12 -> I-ORG
13 -> B-PERCENT
14 -> I-PERCENT
15 -> B-ORDINAL
16 -> B-MONEY
17 -> I-MONEY
18 -> B-WORK_OF_ART
19 -> I-WORK_OF_ART
20 -> B-FAC
21 -> B-TIME
22 -> I-CARDINAL
23 -> B-LOC
24 -> B-QUANTITY
25 -> I-QUANTITY
26 -> I-NORP
27 -> I-LOC
28 -> B-PRODUCT
29 -> I-TIME
30 -> B-EVENT
31 -> I-EVENT
32 -> I-FAC
33 -> B-LANGUAGE
34 -> I-PRODUCT
35 -> I-ORDINAL
36 -> I-LANGUAGE


In [23]:
def convert_tag(original_tag_id):
    # Get the original label, e.g. B-GPE or I-PERSON
    original_label = model.config.id2label[original_tag_id]

    # Keep ordinary non-entity tokens as O
    if original_label == "O":
        return label2id["O"]

    # Split the BIO prefix from the entity type
    prefix, entity = original_label.split("-", 1)

    # If this entity type belongs to our schema, map it
    if entity in label_mapping:
        reconnect_entity = label_mapping[entity]
        reconnect_label = f"{prefix}-{reconnect_entity}"
        return label2id[reconnect_label]

    # Mask unsupported entity types from the training loss
    return -100

In [24]:
test_labels = [
    "O",
    "B-PERSON",
    "I-GPE",
    "B-ORG",
    "B-PRODUCT",
    "B-MONEY",
    "B-CARDINAL",
    "I-FAC",
]

for original_label in test_labels:
    original_id = model.config.label2id.get(original_label)

    # Use the original ID-to-label mapping to avoid relying on label2id ordering
    if original_id is None or model.config.id2label[original_id] != original_label:
        original_id = next(
            idx for idx, label in model.config.id2label.items()
            if label == original_label
        )

    converted_id = convert_tag(original_id)
    converted_label = (
        "-100" if converted_id == -100 else id2label[converted_id]
    )

    print(f"{original_label:15} -> {converted_label}")

O               -> O
B-PERSON        -> B-PERSON
I-GPE           -> I-PLACE
B-ORG           -> B-ORGANIZATION
B-PRODUCT       -> B-ITEM
B-MONEY         -> -100
B-CARDINAL      -> -100
I-FAC           -> I-PLACE


In [25]:


def convert_example(example):
    converted_tags = [
        convert_tag(tag_id)
        for tag_id in example["tags"]
    ]

    return {"ner_tags": converted_tags}


converted_dataset = dataset.map(convert_example)

print(converted_dataset)

Map:   0%|          | 0/59924 [00:00<?, ? examples/s]

Map:   0%|          | 0/8528 [00:00<?, ? examples/s]

Map:   0%|          | 0/8262 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags', 'ner_tags'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['tokens', 'tags', 'ner_tags'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['tokens', 'tags', 'ner_tags'],
        num_rows: 8262
    })
})


In [26]:
for i in range(3):
    example = converted_dataset["train"][i]

    print("\nTokens:", example["tokens"])
    print("Original tags:", [
        model.config.id2label[tag_id]
        for tag_id in example["tags"]
    ])
    print("Converted tags:", [
        "-100" if tag_id == -100 else id2label[tag_id]
        for tag_id in example["ner_tags"]
    ])


Tokens: ['People', 'start', 'their', 'own', 'businesses', 'for', 'many', 'reasons', '.']
Original tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Converted tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Tokens: ['But', 'a', 'chance', 'to', 'fill', 'out', 'sales', '-', 'tax', 'records', 'is', 'rarely', 'one', 'of', 'them', '.']
Original tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-CARDINAL', 'O', 'O', 'O']
Converted tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', '-100', 'O', 'O', 'O']

Tokens: ['Red', 'tape', 'is', 'the', 'bugaboo', 'of', 'small', 'business', '.']
Original tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Converted tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [27]:
def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=256,
    )

    word_ids = tokenized.word_ids()
    original_tags = example["ner_tags"]

    aligned_labels = []
    previous_word_id = None

    for word_id in word_ids:
        # Special tokens have no corresponding original word
        if word_id is None:
            aligned_labels.append(-100)

        # First subword of an original word
        elif word_id != previous_word_id:
            aligned_labels.append(original_tags[word_id])

        # Additional subwords of the same word
        else:
            original_label_id = original_tags[word_id]

            # Keep ignored words ignored
            if original_label_id == -100:
                aligned_labels.append(-100)
            else:
                label = id2label[original_label_id]

                # Convert B-ENTITY to I-ENTITY for continuation subwords
                if label.startswith("B-"):
                    continuation_label = "I-" + label[2:]
                    aligned_labels.append(label2id[continuation_label])
                else:
                    aligned_labels.append(original_label_id)

        previous_word_id = word_id

    tokenized["labels"] = aligned_labels
    return tokenized

In [28]:
sample = converted_dataset["train"][10]

tokenized_sample = tokenize_and_align_labels(sample)

tokens = tokenizer.convert_ids_to_tokens(
    tokenized_sample["input_ids"]
)

for token, label_id in zip(tokens, tokenized_sample["labels"]):
    label = "-100" if label_id == -100 else id2label[label_id]
    print(f"{token:20} {label}")

[CLS]                -100
▁Meanwhile           O
▁,                   O
▁they                O
▁tackle              O
▁the                 O
▁mounds              O
▁of                  O
▁paper               O
▁-                   O
-                    O
▁and                 O
▁fantasize           O
▁about               O
▁a                   O
▁dream               O
▁world               O
▁where               O
▁bulk                O
▁-                   O
▁mail                O
▁postal              O
▁regulations         O
▁and                 O
▁government          O
▁inspectors          O
▁are                 O
▁banished            O
▁.                   O
[SEP]                -100


In [29]:
for example in converted_dataset["train"]:
    if any(tag != 0 and tag != -100 for tag in example["ner_tags"]):
        sample = example
        break

tokenized_sample = tokenize_and_align_labels(sample)
tokens = tokenizer.convert_ids_to_tokens(tokenized_sample["input_ids"])

for token, label_id in zip(tokens, tokenized_sample["labels"]):
    label = "-100" if label_id == -100 else id2label[label_id]
    print(f"{token:20} {label}")


[CLS]                -100
▁Last                B-DATE
▁week                I-DATE
▁,                   O
▁Sen                 O
.                    O
▁Malcolm             B-PERSON
▁Wall                I-PERSON
op                   I-PERSON
▁-                   O
LR                   O
B                    O
-                    O
▁R                   -100
.                    -100
▁,                   O
▁Wyo                 B-PLACE
▁.                   I-PLACE
▁-                   O
RR                   O
B                    O
-                    O
▁held                O
▁hearings            O
▁on                  O
▁a                   O
▁bill                O
▁to                  O
▁strengthen          O
▁an                  O
▁existing            O
▁law                 O
▁designed            O
▁to                  O
▁reduce              O
▁regulatory          O
▁hassles             O
▁for                 O
▁small               O
▁businesses          O
▁.                   O
[SEP]

In [30]:
tokenized_dataset = converted_dataset.map(
    tokenize_and_align_labels,
    batched=False,
    remove_columns=converted_dataset["train"].column_names,
)

print(tokenized_dataset)

Map:   0%|          | 0/59924 [00:00<?, ? examples/s]

Map:   0%|          | 0/8528 [00:00<?, ? examples/s]

Map:   0%|          | 0/8262 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8262
    })
})


In [31]:
print("Training example:")
print(tokenized_dataset["train"][10])

print("\nNumber of training examples:", len(tokenized_dataset["train"]))
print("Number of validation examples:", len(tokenized_dataset["validation"]))
print("Number of test examples:", len(tokenized_dataset["test"]))

Training example:
{'input_ids': [1, 4134, 366, 306, 5574, 262, 41319, 265, 1089, 341, 271, 263, 79444, 314, 266, 2354, 447, 399, 5906, 341, 3447, 15800, 3353, 263, 671, 22315, 281, 43397, 323, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100]}

Number of training examples: 59924
Number of validation examples: 8528
Number of test examples: 8262


In [32]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

model.config.num_labels

[transformers] You passed `num_labels=15` which is incompatible to the `id2label` map of length `37`.


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: tner/deberta-v3-large-ontonotes5
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([37]) vs model:torch.Size([15])            
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([37, 1024]) vs model:torch.Size([15, 1024])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


15

In [33]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")
print("Number of labels:", model.config.num_labels)
print("Label mapping size:", len(model.config.id2label))
print("Labels:", model.config.id2label)

Device: cuda
GPU: Tesla T4
Number of labels: 15
Label mapping size: 15
Labels: {0: 'O', 1: 'B-PERSON', 2: 'I-PERSON', 3: 'B-PLACE', 4: 'I-PLACE', 5: 'B-DATE', 6: 'I-DATE', 7: 'B-TIME', 8: 'I-TIME', 9: 'B-EVENT', 10: 'I-EVENT', 11: 'B-ITEM', 12: 'I-ITEM', 13: 'B-ORGANIZATION', 14: 'I-ORGANIZATION'}


In [34]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import inspect

print("Training tools imported successfully.")
print("TrainingArguments supports eval_strategy:",
      "eval_strategy" in inspect.signature(TrainingArguments).parameters)

Training tools imported successfully.
TrainingArguments supports eval_strategy: True


In [35]:
import transformers
import inspect
from transformers import TrainingArguments

print("Transformers version:", transformers.__version__)

supported_args = inspect.signature(TrainingArguments).parameters

for arg in [
    "warmup_ratio",
    "warmup_steps",
    "eval_strategy",
    "evaluation_strategy",
    "gradient_checkpointing",
    "fp16",
]:
    print(f"{arg}: {arg in supported_args}")

Transformers version: 5.20.2
warmup_ratio: False
warmup_steps: True
eval_strategy: True
evaluation_strategy: False
gradient_checkpointing: True
fp16: True


In [36]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "/content/drive/MyDrive/reconnect-deberta-ner",

    # Training
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # GPU memory and precision
    fp16=True,
   gradient_checkpointing=True,

    # Optimization
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=375,

    # Evaluation and checkpoints
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=50,
    report_to="none",
    seed=42,
)

print("Training configuration created successfully.")
print("Warmup steps:", training_args.warmup_steps)
print("Evaluation strategy:", training_args.eval_strategy)

Training configuration created successfully.
Warmup steps: 375
Evaluation strategy: IntervalStrategy.STEPS


In [37]:

from transformers import Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

print("Trainer is ready.")

Trainer is ready.


In [39]:
train_result = trainer.train()

Step,Training Loss,Validation Loss
500,0.210758,0.077474


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
500,0.210758,0.077474
1000,0.338539,0.083059
1500,0.174871,0.081114
2000,0.245304,0.071085
2500,0.237710,0.073079
3000,0.202345,0.071464
3500,0.219172,0.071687
3746,0.229745,0.072047


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Could not locate the best model at /content/drive/MyDrive/reconnect-deberta-ner/checkpoint-2000/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


In [40]:
trainer.save_model(
    "/content/drive/MyDrive/reconnect-deberta-ner-final"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/reconnect-deberta-ner-final"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/reconnect-deberta-ner-final/tokenizer_config.json',
 '/content/drive/MyDrive/reconnect-deberta-ner-final/tokenizer.json')

In [41]:
import os

for root, dirs, files in os.walk("/content"):
    dirs[:] = [
        d for d in dirs
        if d not in {"drive", ".cache", ".ipynb_checkpoints"}
    ]

    if any(
        name.startswith("checkpoint-")
        or name in {"model.safetensors", "pytorch_model.bin"}
        for name in files
    ):
        print("Found:", root)
        print(files[:10])

In [64]:
import torch

text = "Last Sunday at 3 PM, Maya attended the Dashain Festival at Kathmandu Durbar Square with her family, where she received a Samsung phone from Nepal Red Cross."

trainer.model.eval()

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
device = next(trainer.model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = trainer.model(**inputs)

predictions = outputs.logits.argmax(dim=-1)[0]

for token, label_id in zip(
    tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]),
    predictions.tolist()
):
    label = id2label[label_id]
    if label != "O":
        print(f"{token:20} -> {label}")

▁Last                -> B-DATE
▁Sunday              -> I-DATE
▁3                   -> B-TIME
▁PM                  -> I-TIME
▁Maya                -> B-PERSON
▁the                 -> B-EVENT
▁Dash                -> I-EVENT
ain                  -> I-EVENT
▁Festival            -> I-EVENT
▁Kathmandu           -> B-PLACE
▁Durbar              -> I-PLACE
▁Square              -> I-PLACE
▁Samsung             -> B-ORGANIZATION
▁Nepal               -> B-ORGANIZATION
▁Red                 -> I-ORGANIZATION
▁Cross               -> I-ORGANIZATION


In [43]:
!pip -q install seqeval

In [44]:
import numpy as np
from seqeval.metrics import classification_report, f1_score

# Predict on the existing, held-out test set
test_results = trainer.predict(tokenized_dataset["test"])

predictions = np.argmax(test_results.predictions, axis=2)
true_labels = test_results.label_ids

true_predictions = []
true_references = []

for pred_seq, label_seq in zip(predictions, true_labels):
    pred_labels = []
    reference_labels = []

    for pred_id, label_id in zip(pred_seq, label_seq):
        if label_id == -100:
            # Ignore masked labels during evaluation
            pred_labels.append("O")
            reference_labels.append("O")
        else:
            pred_labels.append(id2label[pred_id])
            reference_labels.append(id2label[label_id])

    true_predictions.append(pred_labels)
    true_references.append(reference_labels)

print("Test-set entity-level F1:", round(
    f1_score(true_references, true_predictions), 4
))

print("\nDetailed classification report:")
print(classification_report(
    true_references,
    true_predictions,
    digits=4,
    zero_division=0
))

Test-set entity-level F1: 0.9195

Detailed classification report:
              precision    recall  f1-score   support

        DATE     0.8580    0.8901    0.8738      1602
       EVENT     0.7051    0.8730    0.7801        63
        ITEM     0.6526    0.8158    0.7251        76
ORGANIZATION     0.9150    0.9181    0.9166      1795
      PERSON     0.9534    0.9567    0.9551      1988
       PLACE     0.9578    0.9507    0.9542      2554
        TIME     0.6607    0.6981    0.6789       212

   micro avg     0.9141    0.9251    0.9195      8290
   macro avg     0.8147    0.8718    0.8405      8290
weighted avg     0.9159    0.9251    0.9203      8290



In [45]:
from pathlib import Path

final_dir = Path(
    "/content/drive/MyDrive/reconnect-deberta-ner-final"
)

print("Model weights:", list(final_dir.glob("*.safetensors")))
print("Model config exists:", (final_dir / "config.json").exists())
print("Tokenizer config exists:", (final_dir / "tokenizer_config.json").exists())

Model weights: [PosixPath('/content/drive/MyDrive/reconnect-deberta-ner-final/model.safetensors')]
Model config exists: True
Tokenizer config exists: True


In [46]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

original_model_id = "tner/deberta-v3-large-ontonotes5"

baseline_tokenizer = AutoTokenizer.from_pretrained(
    original_model_id
)

baseline_model = AutoModelForTokenClassification.from_pretrained(
    original_model_id
)

print("Original checkpoint loaded.")
print("Number of labels:", baseline_model.config.num_labels)
print("Fine-tuned model remains in:", type(trainer.model).__name__)

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Original checkpoint loaded.
Number of labels: 37
Fine-tuned model remains in: DebertaV2ForTokenClassification


In [47]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name
)

ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

In [48]:
text = "Barack Obama was born in Hawaii and later became president of the United States."

results = ner_pipeline(text)

for entity in results:
    print(entity)

{'entity_group': 'LABEL_0', 'score': np.float32(0.50579804), 'word': 'Barack', 'start': 0, 'end': 6}
{'entity_group': 'LABEL_1', 'score': np.float32(0.5445138), 'word': 'Obama', 'start': 6, 'end': 12}
{'entity_group': 'LABEL_0', 'score': np.float32(0.6842428), 'word': 'was born in', 'start': 12, 'end': 24}
{'entity_group': 'LABEL_1', 'score': np.float32(0.6964747), 'word': 'Hawaii', 'start': 24, 'end': 31}
{'entity_group': 'LABEL_0', 'score': np.float32(0.61949366), 'word': 'and later became president of the', 'start': 31, 'end': 65}
{'entity_group': 'LABEL_1', 'score': np.float32(0.6640401), 'word': 'United States', 'start': 65, 'end': 79}
{'entity_group': 'LABEL_0', 'score': np.float32(0.8129416), 'word': '.', 'start': 79, 'end': 80}


In [49]:
finetuned_model = pipeline(
    "ner",
    model="/content/drive/MyDrive/reconnect-deberta-ner-final",
    tokenizer="/content/drive/MyDrive/reconnect-deberta-ner-final",
    aggregation_strategy="simple"
)

print("Original pretrained model:")
print(ner_pipeline(text))

print("\nFine-tuned model:")
print(finetuned_model(text))

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Original pretrained model:
[{'entity_group': 'LABEL_0', 'score': np.float32(0.50579804), 'word': 'Barack', 'start': 0, 'end': 6}, {'entity_group': 'LABEL_1', 'score': np.float32(0.5445138), 'word': 'Obama', 'start': 6, 'end': 12}, {'entity_group': 'LABEL_0', 'score': np.float32(0.6842428), 'word': 'was born in', 'start': 12, 'end': 24}, {'entity_group': 'LABEL_1', 'score': np.float32(0.6964747), 'word': 'Hawaii', 'start': 24, 'end': 31}, {'entity_group': 'LABEL_0', 'score': np.float32(0.61949366), 'word': 'and later became president of the', 'start': 31, 'end': 65}, {'entity_group': 'LABEL_1', 'score': np.float32(0.6640401), 'word': 'United States', 'start': 65, 'end': 79}, {'entity_group': 'LABEL_0', 'score': np.float32(0.8129416), 'word': '.', 'start': 79, 'end': 80}]

Fine-tuned model:
[{'entity_group': 'PERSON', 'score': np.float32(0.99955463), 'word': 'Barack Obama', 'start': 0, 'end': 12}, {'entity_group': 'PLACE', 'score': np.float32(0.99947995), 'word': 'Hawaii', 'start': 24, '

In [55]:
###evaluate the untrained NER-head baseline using the same microsoft/deberta-v3-base backbone

In [51]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    DataCollatorForTokenClassification,
)
import numpy as np
from seqeval.metrics import (
    classification_report,
    f1_score,
)

BASE_MODEL = "microsoft/deberta-v3-base"

# Use the exact same label mapping as your fine-tuned model
label_names = [id2label[i] for i in range(len(id2label))]

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

baseline_model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

print("Loaded original DeBERTa backbone with a fresh NER head.")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

Loaded original DeBERTa backbone with a fresh NER head.


In [53]:
import numpy as np
from seqeval.metrics import (
    f1_score,
    classification_report,
    precision_score,
    recall_score,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        example_predictions = []
        example_labels = []

        for pred_id, label_id in zip(prediction, label):
            # Ignore padding and special tokens
            if label_id == -100:
                continue

            example_predictions.append(id2label[int(pred_id)])
            example_labels.append(id2label[int(label_id)])

        true_predictions.append(example_predictions)
        true_labels.append(example_labels)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

In [56]:
%whos

Variable                             Type                                  Data/Info
------------------------------------------------------------------------------------
AutoModelForTokenClassification      type                                  <class 'transformers.mode<...>lForTokenClassification'>
AutoTokenizer                        type                                  <class 'transformers.mode<...>tion_auto.AutoTokenizer'>
BASE_MODEL                           str                                   microsoft/deberta-v3-base
Counter                              type                                  <class 'collections.Counter'>
DataCollatorForTokenClassification   type                                  <class 'transformers.data<...>rForTokenClassification'>
HfApi                                type                                  <class 'huggingface_hub.hf_api.HfApi'>
LABEL_MAP                            dict                                  n=7
Path                                 t

In [58]:
print("Tokenized dataset:")
print(tokenized_dataset)

print("\nConverted dataset:")
print(converted_dataset)

Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8262
    })
})

Converted dataset:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags', 'ner_tags'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['tokens', 'tags', 'ner_tags'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['tokens', 'tags', 'ner_tags'],
        num_rows: 8262
    })
})


In [59]:
test_dataset = tokenized_dataset["test"]
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 8262
})


In [60]:
print("Dataset variables:")
print("dataset:", dataset)
print("converted_dataset:", converted_dataset)
print("tokenized_dataset:", tokenized_dataset)

Dataset variables:
dataset: DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8262
    })
})
converted_dataset: DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags', 'ner_tags'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['tokens', 'tags', 'ner_tags'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['tokens', 'tags', 'ner_tags'],
        num_rows: 8262
    })
})
tokenized_dataset: DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['

In [61]:
test_dataset = tokenized_dataset["test"]

print("Test examples:", len(test_dataset))

Test examples: 8262


In [62]:
baseline_results = baseline_trainer.evaluate(test_dataset)

print("\nBaseline evaluation results:")
print(baseline_results)

Training Loss,Validation Loss,Step,Precision,Recall,F1
No log,3.078125,0,0.002061,0.033293,0.003882



Baseline evaluation results:
{'eval_loss': 3.078125, 'eval_precision': 0.00206117815748596, 'eval_recall': 0.033293124246079615, 'eval_f1': 0.003882020338410903}


In [63]:
from seqeval.metrics import classification_report
import numpy as np

pred_output = baseline_trainer.predict(test_dataset)

predictions = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

true_predictions = []
true_labels = []

for prediction, label in zip(predictions, labels):
    pred_tags = []
    true_tags = []

    for pred_id, label_id in zip(prediction, label):
        if label_id == -100:
            continue

        pred_tags.append(id2label[int(pred_id)])
        true_tags.append(id2label[int(label_id)])

    true_predictions.append(pred_tags)
    true_labels.append(true_tags)

print(classification_report(true_labels, true_predictions, digits=4))

              precision    recall  f1-score   support

        DATE     0.0005    0.0056    0.0009      1602
       EVENT     0.0003    0.1746    0.0007        63
        ITEM     0.0002    0.0658    0.0003        76
ORGANIZATION     0.0056    0.1359    0.0107      1795
      PERSON     0.0026    0.0005    0.0008      1988
       PLACE     0.0011    0.0023    0.0015      2554
        TIME     0.0000    0.0000    0.0000       212

   micro avg     0.0021    0.0333    0.0039      8290
   macro avg     0.0015    0.0550    0.0021      8290
weighted avg     0.0023    0.0333    0.0032      8290



In [68]:
import torch

text = "Yesterday Binu told me She can't come with me to my Khwopa college at 3pm because of her busy schedule as our work collides with her ABC company"
trainer.model.eval()

inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
device = next(trainer.model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = trainer.model(**inputs)

predictions = outputs.logits.argmax(dim=-1)[0]

for token, label_id in zip(
    tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]),
    predictions.tolist()
):
    label = id2label[label_id]
    if label != "O":
        print(f"{token:20} -> {label}")

▁Yesterday           -> B-DATE
▁Bin                 -> B-PERSON
u                    -> I-PERSON
▁Kh                  -> B-PLACE
wop                  -> I-PLACE
a                    -> I-PLACE
▁3                   -> B-TIME
pm                   -> I-TIME
▁ABC                 -> B-ORGANIZATION
